### **Dataset Generation**

In [1]:
from gen_dataset import simulate_imgs

GEN_DATASET: bool = False

dirpath: str = '/home/edoardo/Desktop/ImgsMockDatasetDMs'
# dirpath: str = '/mnt/d/DL_test_dataset'

if GEN_DATASET:
    num_imgs: int = -1
    polygonVertices: int = 6

    simulate_imgs(
        num_imgs=num_imgs,
        polygonVertices=polygonVertices,
        save_to_dir=dirpath,
        start_num=0,
    )
else:
    print('Dataset already generated!')

Dataset already generated!


### **Dataset Handling**

In [2]:
from typing import Any, Callable
from pathlib import Path
from PIL import Image
import random

from tqdm import tqdm
import torch
from torch.types import Tensor
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms as ts
from torchvision.transforms import Compose

In [3]:
def get_data_filespaths(dirpath: str | Path, data_frmt: str = 'png', shuffle: bool = False) -> list[str]:
    """Groups all the data file-paths (of the given format) inside the specified directory."""
    dirpath_ = Path(dirpath)
    paths_list: list[str] = [str(path) for path in dirpath_.glob(f'*.{data_frmt}')]
    if shuffle: random.shuffle(paths_list)
    return paths_list

def process_data(
    data_paths: list[str | Path],
    open_with: Callable,
    transform: Compose | None,
) -> list[Tensor]:
    """
    Processes the given data by first opening the file and then applying a list of PyTorch
    `Transform` objs or custom-made Callable objs (if `None`, default is `PILToTensor`).
    """
    transform_ = transform if transform is not None else ts.PILToTensor()
    processed = [transform_(open_with(f)) for f in tqdm(data_paths, desc='Processing data')]
    return processed


def get_dataloaders(
    dataset: Dataset,
    batch_size: int,
    valid_size: float = 0.0,
    **kwargs,
) -> tuple[DataLoader, DataLoader | None]:
    """Training and Validation dataset loaders generation."""
    if valid_size and not (0.0 <= valid_size < 1.0):
        raise ValueError(f"Invalid 'valid_size' value {valid_size}, must be in [0, 1).")
    
    print('Baking the dataset...')    
    if valid_size > 0:
        vlen = int(valid_size * len(dataset))
        tlen = len(dataset) - vlen
        train, valid = random_split(dataset, [tlen, vlen])
        train_dl = DataLoader(train, batch_size, shuffle=True, **kwargs)
        valid_dl = DataLoader(valid, batch_size, shuffle=False, **kwargs)
    else:
        train_dl = DataLoader(dataset, batch_size, shuffle=True, **kwargs)
        valid_dl = None
    print('Dataset ready-to-go!')

    return train_dl, valid_dl


def save_dataset(
    dataset: Dataset,
    save_to: str | Path,
    overwrite: bool = False,
    **kwargs,
) -> None:
    """
    Saves given dataset to '.pt' file.
    """
    if Path(save_to).exists() and not overwrite:
        print("Dataset already saved!")
        return
    print("Saving dataset...")
    torch.save(dataset, save_to, **kwargs)
    print("Dataset saved!")
    return

def load_dataset(filepath: str | Path, **kwargs) -> Dataset:
    """
    Load given dataset from '.pt' file.
    """
    print("Loading dataset...")
    dataset = torch.load(filepath, weights_only=False, **kwargs)
    print("Dataset loaded!")
    return dataset

**Tensor Operations**

In [4]:
def center_tensor(tensor: Tensor, eps: float = 1e-8) -> Tensor:
    """
    Centers given tensor of shape `[C, H, W]` per-channel
    by subtracting the mean and dividing by the std.
    """
    dims = (1, 2)
    tensor_ = tensor.to(torch.float)
    mu = tensor_.mean(dims, keepdim=True)
    std = tensor_.std(dims, keepdim=True)
    centered = (tensor_ - mu) / (std + eps)
    return centered

def normalise(tensor: Tensor, norm_range: str = 'unilateral') -> Tensor:
    """Normalises given tensor in the range [0, 1] or [-1, 1]."""
    if norm_range not in ['unilateral', 'bilateral']:
        raise ValueError(f"Invalid 'norm_range' {norm_range}.")
    
    normalised = (tensor - tensor.min()) / (tensor.max() - tensor.min())
    if norm_range == 'bilateral':
        normalised = 2 * normalised - 1.0

    return normalised

**Dataset Realisation**

In [5]:
class ImageDataset(Dataset):
    """
    Baseline AutoEncoder dataset configurator.
    """
    def __init__(self, data_path: str | Path, transform: Compose | None) -> None:
        # transform: Callable = (
        #     preprocess if preprocess is not None else lambda x: ts.PILToTensor()(x)
        # )
        # self.data: list[Tensor] = [
        #     transform(Image.open(img).convert('RGB')) for img in tqdm(files_list)
        # ]
        files_list = get_data_filespaths(data_path, shuffle=True)
        self.data = process_data(files_list, lambda img: Image.open(img).convert('RGB'), transform)
    
    def __len__(self) -> int:
        return len(self.data)
    
    def __getitem__(self, index) -> tuple[Tensor, Tensor]:
        sample = self.data[index]
        return sample, sample

In [6]:
BASEPATH: str = '/home/edoardo/Desktop/MockDataForDMs'
DATASET_ID: str = 'mockPolyImgsDataset'
BATCH_SIZE: int = 250
VALID_SIZE: float = 0.2
PREPROCESSING: Compose = ts.Compose(
    [
        ts.functional.to_grayscale,
        ts.PILToTensor(),
        ts.Resize((36, 36), antialias=True),
        center_tensor,
        normalise,
    ]
)

In [7]:
def handle_dataset(*filepaths: str | Path, **kwargs) -> tuple[DataLoader, DataLoader | None]:
    """Handles dataset(s). A dataset is loaded if its file exists, else is generated and saved."""
    raise NotImplementedError


try:
    dataset: Dataset = load_dataset(f'{BASEPATH}/{DATASET_ID}.pt')
except FileNotFoundError:
    print('Dataset not found, generating...')
    dataset: ImageDataset = ImageDataset(f'{BASEPATH}/ImgsMockDatasetDMs', PREPROCESSING)
    save_dataset(dataset, f'{BASEPATH}/{DATASET_ID}.pt')

train_dl, valid_dl = get_dataloaders(dataset, BATCH_SIZE, VALID_SIZE)

Loading dataset...
Dataset loaded!
Baking the dataset...
Dataset ready-to-go!


### **Model Definition**

In [8]:
import torch.nn as nn

def get_conv_block(
    in_dims: int,
    out_dims: int,
    kernel_size: int,
    padding: int,
) -> nn.ModuleList:
    """Defines baseline conv block for mock model."""
    block = [
        nn.Conv2d(in_dims, out_dims, kernel_size, padding=padding),
        nn.BatchNorm2d(out_dims),
        nn.ReLU(),
    ]
    return nn.ModuleList(block)


class MockModel(nn.Module):
    """
    Baseline mock CNN model for `spark` API tests.
    """
    def __init__(
        self,
        data_shape: torch.Size,
        out_features: int,
        maxpool: int = 2,
        dropout: float = 0.3, 
    ) -> None:
        super().__init__()
        in_dim = int(data_shape[0])
        in_features = int(data_shape.numel() / maxpool)
        self.net = nn.Sequential(
            *get_conv_block(in_dim, 16, 7, 3),
            *get_conv_block(16, 16, 5, 2),
            *get_conv_block(16, in_dim, 3, 1),
            nn.MaxPool2d(maxpool),
            nn.Flatten(),
            nn.Linear(in_features, 1024), nn.ReLU(), nn.Dropout(p=dropout),
            nn.Linear(1024, out_features),
        )
    
    def forward(self, x: Tensor) -> Tensor:
        out = self.net(x)
        return out

### **Model Handling**

In [9]:
from typing import OrderedDict

def save_model(
    state_dict: OrderedDict,
    save_to: str | Path,
    info: dict[str, Any] | None = None,
    overwrite: bool = False,
    **kwargs,
) -> None:
    """Saves given model and its state to '.pt' file, plus other info."""
    if Path(save_to).exists() and not overwrite:
        print("Model already saved!")
        return
    print("Saving model...")
    data = info if info is not None else {}
    data['state_dict'] = state_dict
    torch.save(data, save_to, **kwargs)
    print("Model saved!")
    return

def load_model(filepath: str | Path, **kwargs) -> dict[str, Any]:
    """Load given model, its state and saved info from '.pt' file."""
    print("Loading model...")
    model_state: dict = torch.load(filepath, weights_only=False, **kwargs)
    print("Model loaded!")
    return model_state




def save_checkpoint() -> None:
    """Saves a checkpoint during training, given a metric."""
    raise NotImplementedError

In [15]:
MODEL_ID: str = 'mockCNNbasicModel'
DATA_SHAPE: torch.Size = torch.Size([1, 36, 36])
OUT_FEATURES: int = 4

try:
    model_data: dict[str, Any] = load_model(f'{BASEPATH}/{MODEL_ID}.pt')
    model: nn.Module = model_data['state_dict']
except FileNotFoundError:
    print('Model not found, generating...')
    model: nn.Module = MockModel(data_shape=DATA_SHAPE, out_features=OUT_FEATURES)
    save_model(model, f'{BASEPATH}/{MODEL_ID}.pt')

print(model)

Loading model...
Model loaded!
MockModel(
  (net): Sequential(
    (0): Conv2d(1, 16, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3))
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(16, 16, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): Conv2d(16, 1, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): BatchNorm2d(1, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ReLU()
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Flatten(start_dim=1, end_dim=-1)
    (11): Linear(in_features=648, out_features=1024, bias=True)
    (12): ReLU()
    (13): Dropout(p=0.3, inplace=False)
    (14): Linear(in_features=1024, out_features=4, bias=True)
  )
)


In [12]:
###### -------------------  TESTING TORCH OPs ------------------------ ######
assert False

from PIL import Image
import numpy as np
from numpy.typing import NDArray
import torch
from torch.types import Tensor
from torchvision import transforms as ts
import torchvision.transforms.functional as F

def get_dataset(
    data_path: str | Path,
    batch_size: int,
    configurator: Callable[[list[str]], Dataset],
    valid_size: float | None = None,
    data_frmt: str = 'png',
    shuffle: bool = True,
) -> tuple[DataLoader, DataLoader | None]:
    """Dataset generation with given data pre-processing."""
    if valid_size and not (0 <= valid_size < 1):
        raise ValueError(f"Invalid 'valid_size' value {valid_size}, must be in [0, 1).")

    # load data files paths
    paths_list = get_data_filespaths(data_path, data_frmt, shuffle)
    # config dataset
    print('Baking the dataset...')
    dataset = configurator(paths_list)
    # split dataset in train/validation (if given `valid_size`)
    if valid_size is not None:
        v = int(valid_size * len(dataset))
        validation = Subset(dataset, torch.arange(v))
        training = Subset(dataset, torch.arange(v, len(dataset)))
        valid_dataset = DataLoader(validation, batch_size, shuffle=False)
        train_dataset = DataLoader(training, batch_size, shuffle=True)
    else:
        valid_dataset = None
        train_dataset = DataLoader(dataset, batch_size, shuffle=True)
    print('Dataset ready-to-go!')

    return train_dataset, valid_dataset


try:
    train_ds, valid_ds = map(load_dataset, (f'{BASEPATH}/train_DS.pt', f'{BASEPATH}/valid_DS.pt'))
except FileNotFoundError:
    print('No dataset(s) found :c...\n')
    train_ds, valid_ds = get_dataset(
        f'{BASEPATH}/ImgsMockDatasetDMs', BATCH_SIZE, lambda x: ImageDataset(x, PREPROCESS), VALID_SIZE,
    )
    save_dataset(train_ds, save_to=f'{BASEPATH}/train_DS.pt')
    save_dataset(valid_ds, save_to=f'{BASEPATH}/valid_DS.pt')


# ------------------------------------------------------------------------------------------------------------------- #


def np_to_torch(x: NDArray) -> Tensor:
    return F.to_tensor(x)

def pil_to_torch(img: Image) -> Tensor:
    return F.pil_to_tensor(img)


filepath: str = '/home/edoardo/Desktop/MockDataForDMs/ImgsMockDatasetDMs'
target: str = 'circle0'

img = Image.open(f'{filepath}/{target}.png').convert('RGB')
x = np.random.uniform(0, 1, (10, 10))

print(
    torch.allclose(pil_to_torch(img), ts.PILToTensor()(img)),
    torch.allclose(np_to_torch(x), ts.ToTensor()(x)),
    torch.allclose(np_to_torch(x), torch.tensor(x)),
    #torch.allclose(ts.PILToTensor()(img), ts.ToTensor()(img)),
)
img.close()

a = ['fwefwefqwef.npy', 'fwefhu2ewhf.png', 'dhqwiuedh2iuf.pt']

frmt = '*.npy'

b = frmt in a
b

AssertionError: 